In [1]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

# Loading data

In [2]:
learn_data = pd.read_csv("minimal_train_fs.csv", header = None)
learn_data.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female', 'Target']
learn_data["Female"] = learn_data["Female"].astype("category")
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female,Target
0,48,1.504077,5.641907,4.304065,2.4,0.52,0.511111,0,0
1,39,0.641854,5.192957,4.127134,4.3,1.38,0.473684,0,0
2,23,0.000000,5.356586,4.382027,3.1,1.00,0.300000,0,0
3,42,-0.356675,5.023881,4.394449,3.2,1.06,0.285714,1,0
4,54,3.117950,6.324359,3.610918,3.4,0.80,0.504425,1,0


In [3]:
learn_data.isna().value_counts()

Age    TB     Alkphos  Sgot   ALB    AR     BilRatio  Female  Target
False  False  False    False  False  False  False     False   False     449
Name: count, dtype: int64

In [4]:
X = learn_data.drop(columns = ["Target"])
y = learn_data["Target"]

# Metrics

In [5]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

# The classifiers, by themselves
## QDA

In [6]:
QDA_pipeline = Pipeline([('scaler', StandardScaler()), ('QDA', QuadraticDiscriminantAnalysis(priors = (0.5, 0.5)))])

n = 10
regs = np.logspace(start = -4, stop = -0.5, num = 100)

QDA_search = GridSearchCV(estimator = QDA_pipeline,
                          param_grid = {'QDA__reg_param' : regs},
                          scoring = 'f1_macro',
                          cv = 5)
QDA_search.fit(X, y)
QDA_search.best_params_

{'QDA__reg_param': 0.0001}

In [7]:
QDA_search.best_score_

0.6271432896385213

In [8]:
QDA_reg_param_ = QDA_search.best_params_['QDA__reg_param']
qda_model = QuadraticDiscriminantAnalysis(priors = (0.5, 0.5),
                                          reg_param = QDA_reg_param_)
qda_pipeline = Pipeline([('scaler', StandardScaler()), ('QDA', qda_model)])

cross_val_results = pd.DataFrame(cross_validate(qda_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["QDA", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
QDA,0.627143,0.701966,0.665863,0.637054


## Logistic Regression

In [9]:
logreg_pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression(class_weight = "balanced"))])

n = 100
m = 10
Cs = np.logspace(start = -4, stop = 2, num = n)

logreg_search = GridSearchCV(estimator = logreg_pipeline,
                             param_grid = {'logreg__C' : Cs},
                             scoring = 'f1_macro',
                             cv = 5)
logreg_search.fit(X, y)
logreg_search.best_params_

{'logreg__C': 0.6579332246575682}

In [10]:
logreg_search.best_score_

0.6635322595910054

In [11]:
logreg_C = logreg_search.best_params_["logreg__C"]

logreg_model_best = LogisticRegression(C = logreg_C,
                                       class_weight = "balanced")
logreg_pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', logreg_model_best)])

cross_val_results = pd.DataFrame(cross_validate(logreg_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["LogReg-Best", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.663532,0.724197,0.681851,0.679451
QDA,0.627143,0.701966,0.665863,0.637054


## Gaussian kernel SVC

In [12]:
rbfsvc = SVC(kernel = "rbf", class_weight = "balanced", probability = True)
rbfsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", rbfsvc)])

n = 50
m = 50
Cs = np.logspace(start = -1, stop = 2, num = n)
gammas = np.logspace(start = -1, stop = 1, num = m) / X.shape[0]

rbfsvc_search = GridSearchCV(estimator = rbfsvc_pipeline,
                             param_grid = {'svc__C' : Cs,
                                           'svc__gamma' : gammas},
                             scoring = 'f1_macro',
                             cv = 5)
rbfsvc_search.fit(X, y)
rbfsvc_search.best_params_

KeyboardInterrupt: 

In [ ]:
rbfsvc_search.best_score_

0.660344012069942

In [ ]:
rbfsvc_C = rbfsvc_search.best_params_['svc__C']
rbfsvc_gamma = rbfsvc_search.best_params_['svc__gamma']
rbfsvc_best = SVC(kernel = "rbf", C = rbfsvc_C, gamma = rbfsvc_gamma,
                  class_weight = "balanced", probability = True)
rbfsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", rbfsvc_best)])

cross_val_results = pd.DataFrame(cross_validate(rbfsvc_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Gaussian SVC", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Voting,0.676857,0.732529,0.68851,0.695031
LogReg-Best,0.663532,0.724197,0.681851,0.679451
Gaussian SVC,0.660344,0.702813,0.667546,0.68392
Random Forest,0.650032,0.703851,0.665716,0.668215
QDA,0.627143,0.701966,0.665863,0.637054
AdaBoost,0.615167,0.618851,0.618117,0.686042


## Random Forest

In [ ]:
from imblearn.pipeline import Pipeline as PipelineIMB
from imblearn.under_sampling import RandomUnderSampler

rf_pipeline = PipelineIMB([('undersamp', RandomUnderSampler()),
                           ('scaler', StandardScaler()),
                           ('rf', RandomForestClassifier(class_weight = "balanced"))])

criteria = ["gini", "entropy", "log_loss"]
max_features = ["sqrt", "log2", None]
depths = [2, 4, 6, 8, 10]
ns = [100]
min_samples = [5, 10, 20, 30, 40, 50]
oob_score = [True]

rf_search = GridSearchCV(estimator = rf_pipeline,
                         param_grid = {'rf__criterion' : criteria,
                                      'rf__max_features' : max_features,
                                      'rf__max_depth' : depths,
                                      'rf__min_samples_split' : min_samples,
                                      'rf__n_estimators' : ns,
                                      'rf__oob_score' : oob_score},
                         scoring = 'f1_macro',
                         cv = 5)
rf_search.fit(X, y)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('undersamp', RandomUnderSampler()),
                                       ('scaler', StandardScaler()),
                                       ('rf',
                                        RandomForestClassifier(class_weight='balanced'))]),
             param_grid={'rf__criterion': ['gini', 'entropy', 'log_loss'],
                         'rf__max_depth': [2, 4, 6, 8, 10],
                         'rf__max_features': ['sqrt', 'log2', None],
                         'rf__min_samples_split': [5, 10, 20, 30, 40, 50],
                         'rf__n_estimators': [100], 'rf__oob_score': [True]},
             scoring='f1_macro')

In [ ]:
rf_search.best_score_

0.6745775485869345

In [ ]:
rf_search.best_params_

{'rf__criterion': 'entropy',
 'rf__max_depth': 10,
 'rf__max_features': 'log2',
 'rf__min_samples_split': 10,
 'rf__n_estimators': 100,
 'rf__oob_score': True}

In [ ]:
rf_criterion = rf_search.best_params_['rf__criterion']
rf_max_depth = rf_search.best_params_['rf__max_depth']
rf_max_features = rf_search.best_params_['rf__max_features']
rf_min_samples_split = rf_search.best_params_['rf__min_samples_split']
rf_n_estimators = rf_search.best_params_['rf__n_estimators']
rf_best = RandomForestClassifier(criterion = rf_criterion,
                                max_depth = rf_max_depth,
                                max_features = rf_max_features,
                                min_samples_split = rf_min_samples_split,
                                n_estimators = rf_n_estimators,
                                class_weight = "balanced")
rf_pipeline = PipelineIMB([("undersamp", RandomUnderSampler()),
                            ("scaler", StandardScaler()),
                            ('rf', rf_best)])

cross_val_results = pd.DataFrame(cross_validate(rf_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Random Forest", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.663532,0.724197,0.681851,0.679451
Gaussian SVC,0.660344,0.702813,0.667546,0.68392
Random Forest,0.650032,0.703851,0.665716,0.668215
QDA,0.627143,0.701966,0.665863,0.637054


# Voting classifiers

The classifiers, by themselves, have similar performance. However, when predicting the training data they show to have different opinions. A consensus should be established.

In [ ]:
qda_pipeline.fit(X, y)
qda_labels = qda_pipeline.predict(X)
logreg_pipeline.fit(X, y)
logreg_labels = logreg_pipeline.predict(X)
rbfsvc_pipeline.fit(X, y)
rbfsvc_labels = rbfsvc_pipeline.predict(X)
rf_pipeline.fit(X, y)
rf_labels = rf_pipeline.predict(X)

In [ ]:
pd.Series(np.logical_and(qda_labels == logreg_labels,
                         qda_labels == rbfsvc_labels,
                         qda_labels == rf_labels)).value_counts()

True     364
False     85
Name: count, dtype: int64

Let's make them vote, then.

In [ ]:
from imblearn.pipeline import Pipeline as PipelineIMB

estimators = [("logreg", logreg_pipeline), ("rbfsvc", rbfsvc_pipeline), ("rf", rf_pipeline)]
votingclass = VotingClassifier(estimators = estimators)

n = 10
weights = [(p1 / n, p2 / n, 1 - p1 / n - p2 / n)
           for p1 in range(0, n + 1)
           for p2 in range(0, n + 1 - p1)]
modes = ["soft", "hard"]

vote_search = GridSearchCV(estimator = votingclass,
                           param_grid = {"weights" : weights,
                                         "voting" : modes},
                           scoring = "f1_macro",
                           cv = 5)
vote_search.fit(X, y)
vote_search.best_params_

{'voting': 'soft', 'weights': (0.7, 0.2, 0.10000000000000003)}

In [ ]:
vote_search.best_score_

0.6801358366494552

In [ ]:
selected_estimators = [("logreg", logreg_pipeline), ("rbfsvc", rbfsvc_pipeline), ("rf", rf_pipeline)]
vote_mode = vote_search.best_params_['voting']
vote_weights = vote_search.best_params_['weights']
vote_best = VotingClassifier(estimators = selected_estimators,
                             voting = vote_mode,
                             weights = vote_weights)

cross_val_results = pd.DataFrame(cross_validate(vote_best, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Voting", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Voting,0.683485,0.728212,0.687024,0.706217
LogReg-Best,0.663532,0.724197,0.681851,0.679451
Gaussian SVC,0.660344,0.702813,0.667546,0.68392
Random Forest,0.650032,0.703851,0.665716,0.668215
QDA,0.627143,0.701966,0.665863,0.637054
AdaBoost,0.615167,0.618851,0.618117,0.686042


### Making predictions

In [ ]:
test_data = pd.read_csv("minimal_test_fs.csv", header = None)
test_data.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female']
test_data["Female"] = learn_data["Female"].astype("category")
test_data.head()

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female
0,11,-0.356675,6.383507,3.367296,4.2,1.40,0.142857,0
1,62,0.587787,5.411646,5.043425,4.0,0.80,0.500000,0
2,60,-0.356675,5.159055,2.639057,4.2,1.10,0.285714,0
3,60,1.740466,5.365976,6.745236,3.2,0.78,0.491228,1
4,48,-0.105361,5.164786,3.988984,2.7,0.90,0.222222,1


In [ ]:
test_y = pd.read_csv("test_y.csv").iloc[:, 1]
test_y

0      1
1      0
2      1
3      0
4      1
      ..
111    1
112    0
113    1
114    0
115    0
Name: Label, Length: 116, dtype: int64

In [ ]:
vote_best.fit(X, y)

labels_vote = pd.DataFrame(columns = ['ID', 'Label'])
labels_vote['Label'] = pd.DataFrame(vote_best.predict(test_data))
labels_vote['ID'] = labels_vote.index + 1
labels_vote.to_csv('new_predictions/vote_best_fs.csv', index = False)
labels_vote

,ID,Label
0,1,1
1,2,0
2,3,1
3,4,0
4,5,0
...,...,...
111,112,0
112,113,0
113,114,1
114,115,0


In [ ]:
compute_metrics(test_y, labels_vote['Label'])

[0.6870247731174883,
 0.7190580503833516,
 0.6838235294117647,
 0.7155172413793104]

# Bagging Classifier

In [ ]:
from sklearn.ensemble import BaggingClassifier

bagger = BaggingClassifier(estimator = vote_pipeline, max_samples = 0.7, n_estimators = 100)

cross_val_results = pd.DataFrame(cross_validate(bagger, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Bagging del Voting", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Voting,0.683485,0.728212,0.687024,0.706217
LogReg-Best,0.663532,0.724197,0.681851,0.679451
Gaussian SVC,0.660344,0.702813,0.667546,0.68392
Bagging del Voting,0.656914,0.702351,0.664197,0.679451
Random Forest,0.650032,0.703851,0.665716,0.668215
Boost,0.649284,0.67613,0.652482,0.68392
QDA,0.627143,0.701966,0.665863,0.637054
AdaBoost,0.536495,0.545216,0.604452,0.703995


# AdaBoost Classifier

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, cross_validate

logreg_model_best = LogisticRegression(C = 0.6579332246575682,
                                       class_weight = "balanced")
rbfsvc_best = SVC(kernel = "rbf",
                  C = 1.4563484775012436,
                  gamma = 0.0002687734166234585,
                  class_weight = "balanced",
                  probability = True)
rf_best = RandomForestClassifier(criterion = "entropy",
                                 max_depth = 10,
                                 max_features = "log2",
                                 min_samples_split = 10,
                                 n_estimators = 50,
                                 class_weight = "balanced")

estimators = [("logreg", logreg_model_best), ("rbfsvc", rbfsvc_best), ("rf", rf_best)]
votingclass = VotingClassifier(estimators = estimators, voting = "soft")
adaboost = AdaBoostClassifier(estimator = votingclass, n_estimators = 25)
boost_pipeline = Pipeline([("scaler", StandardScaler()),
                           ("boost", adaboost)])

cross_val_results = pd.DataFrame(cross_validate(boost_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["AdaBoost", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
AdaBoost,0.532107,0.542611,0.576104,0.697203


In [ ]:
n = 30
learning_rates = np.logspace(start = -2, stop = 1, num = n)
n_estimators = [25]

ada_search = GridSearchCV(estimator = boost_pipeline,
                          param_grid = {"boost__n_estimators" : n_estimators,
                                        "boost__learning_rate" : learning_rates},
                          cv = 5,
                          scoring = "f1_macro")
ada_search.fit(X, y)
ada_search.best_params_

KeyboardInterrupt: 

In [ ]:
ada_search.best_score_

In [ ]:
adaboost = AdaBoostClassifier(estimator = votingclass, learning_rate = 0.28, n_estimators = 100, random_state = 42)
boost_pipeline = Pipeline([("scaler", StandardScaler()),
                           ("boost", adaboost)])

cross_val_results = pd.DataFrame(cross_validate(boost_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values